# RAG vs CAG: Same Model, Two Architectures, Completely Different Approaches

**Author:** Anubhav Singhmaar

A few days ago, I built a RAG pipeline to fix LLM hallucinations. It worked.
Then I read about **CAG (Cache Augmented Generation)** — a technique that skips retrieval entirely and preloads ALL your knowledge into the model's context window.

So I built both. Same model. Same knowledge. Same questions.

### Stack (100% Free)
- **Gemini 2.0 Flash** — 1M token context window
- **ChromaDB** — lightweight vector store for RAG
- **Sentence Transformers** — all-MiniLM-L6-v2 for embeddings
- **Google Colab T4 GPU**

In [ ]:
!pip install -q google-genai chromadb sentence-transformers

## Step 1: Gemini API Key
Get your free key from [Google AI Studio](https://aistudio.google.com/apikey)

In [ ]:
from google.colab import userdata
import os

try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print('API key loaded from Colab Secrets')
except Exception:
    GEMINI_API_KEY = 'YOUR_API_KEY_HERE'
    print('Using hardcoded API key. Consider using Colab Secrets.')

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

## Step 2: Knowledge Base

This is a **fictional internal product document** for a startup called NovaMind.
The model has never seen this. It cannot answer from training data.
If the model gets it right, it came from our pipeline. If wrong, it hallucinated.

In [ ]:
KNOWLEDGE_BASE = """
DOCUMENT: Product Specification — NovaMind AI Assistant Platform (Internal)
===========================================================================

COMPANY OVERVIEW:
NovaMind Technologies is a Series B startup founded in 2024, headquartered in
Bangalore, India. The company builds enterprise AI assistants for the healthcare
and pharmaceutical industry. CEO: Priya Venkatesh. CTO: Arjun Malhotra.
Total funding: $42M across Seed, Series A, and Series B rounds.
Lead investors: Peak XV Partners (Series B), Lightspeed India (Series A).

PRODUCT: NovaMind Assist v3.2
NovaMind Assist is an AI copilot designed for pharmaceutical sales representatives.
It helps reps prepare for doctor visits by summarizing drug interaction data,
generating personalized talking points, and flagging compliance risks in real time.

KEY FEATURES:
1. Drug Interaction Engine: Cross references 14,000+ drug compounds from the
   NovaMind Pharma Graph database. Alerts reps when a doctor's known prescriptions
   may conflict with the product being pitched. Updated weekly from FDA and EMA feeds.

2. Visit Prep Mode: Generates a 2 minute briefing before each doctor meeting.
   Includes the doctor's recent publications, prescription patterns (anonymized),
   and suggested conversation angles based on therapeutic area.

3. Compliance Shield: Monitors rep conversations in real time (with consent) and
   flags statements that may violate FDA promotional guidelines. Accuracy: 94.2%
   on internal benchmarks. False positive rate: 3.1%.

4. Territory Analytics Dashboard: Tracks rep performance across assigned territories.
   Metrics include call frequency, conversion rate, and share of voice vs competitors.

PRICING:
Enterprise tier: $180 per rep per month (minimum 50 seats).
Growth tier: $120 per rep per month (minimum 200 seats, annual contract).
Custom tier: Available for deployments over 1000 seats. Includes dedicated
success manager and custom model fine tuning.

TECHNICAL ARCHITECTURE:
Backend: Python (FastAPI) running on AWS EKS. Primary LLM: Claude Sonnet for
generation tasks, with a fallback to Gemini Flash for high volume, low complexity
queries. Embedding model: Cohere embed v3 for the Drug Interaction Engine.
Vector store: Pinecone (Serverless). Auth: Okta SSO with RBAC.
Average API latency: 340ms (p50), 890ms (p95). Uptime SLA: 99.95%.

RECENT MILESTONES:
March 2026: Signed Cipla as flagship enterprise customer (800 reps).
February 2026: Launched Compliance Shield v2 with real time monitoring.
January 2026: Closed Series B at $28M led by Peak XV Partners.
November 2025: Won Best AI Startup at TechSparks Bangalore.

COMPETITIVE LANDSCAPE:
Primary competitors: Veeva Vault (legacy, CRM focused, weak AI layer),
Aktana (strong analytics, limited generative AI capabilities),
Medidata AI (clinical trial focused, not sales enablement).
NovaMind's moat: Only platform combining real time compliance monitoring
with generative visit preparation. Patent pending on the Compliance Shield
architecture (Indian Patent Application No. 202641023847).

KNOWN LIMITATIONS:
1. Drug Interaction Engine currently covers US and EU markets only.
   India and APAC expansion planned for Q3 2026.
2. Compliance Shield does not yet support languages other than English.
   Hindi and Mandarin support in beta, expected GA by August 2026.
3. Visit Prep Mode requires minimum 6 months of prescription data for
   accurate doctor profiling. New territories show degraded performance
   for the first quarter after deployment.

TEAM:
Total headcount: 87 employees.
Engineering: 42 (including 12 ML engineers).
Product: 8. Design: 5. Sales: 18. Customer Success: 9. Operations: 5.
Key hires in Q1 2026: VP of Engineering (ex Flipkart), Head of Compliance
(ex Pfizer India).
"""

print(f'Knowledge base loaded: {len(KNOWLEDGE_BASE.split())} words')
print('This is a FICTIONAL company. The model has never seen this data.')

## Step 3: Test Questions

In [ ]:
TEST_QUESTIONS = [
    'What is NovaMind Assist and what does it do?',
    'What is the pricing for the Growth tier?',
    'Who are NovaMind main competitors and what is their competitive advantage?',
    'What are the known limitations of the platform?',
    'What is the technical architecture behind NovaMind Assist?',
    'How accurate is Compliance Shield and what is its false positive rate?',
]

print(f'{len(TEST_QUESTIONS)} test questions loaded')

---
# ARCHITECTURE 1: RAG (Retrieval Augmented Generation)

Split doc into chunks → embed → store in vector DB → for each question retrieve top chunks → send to LLM.

**The model never sees the full document.**

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
import time

def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks

chunks = chunk_text(KNOWLEDGE_BASE)
print(f'Document split into {len(chunks)} chunks')
for i, chunk in enumerate(chunks):
    print(f'\nChunk {i+1} ({len(chunk.split())} words):')
    print(chunk[:200] + '...' if len(chunk) > 200 else chunk)

In [ ]:
print('Loading embedding model...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model loaded')

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection('rag_vs_cag')
except:
    pass

collection = chroma_client.create_collection(
    name='rag_vs_cag',
    metadata={'hnsw:space': 'cosine'}
)

print('Embedding and storing chunks...')
embeddings = embed_model.encode(chunks).tolist()
collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f'chunk_{i}' for i in range(len(chunks))]
)
print(f'{len(chunks)} chunks embedded and stored in ChromaDB')

In [ ]:
from google import genai

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_NAME = 'gemini-3-flash-preview'

def rag_query(question, top_k=3):
    start_time = time.time()
    query_embedding = embed_model.encode([question]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)
    retrieved_chunks = results['documents'][0]
    retrieval_time = time.time() - start_time

    context = '\n\n---\n\n'.join(retrieved_chunks)
    prompt = f'You are a helpful assistant. Answer the question based ONLY on the provided context. If the context does not contain the answer, say I dont have enough information.\n\nCONTEXT:\n{context}\n\nQUESTION: {question}\n\nANSWER:'

    response = gemini_client.models.generate_content(model=MODEL_NAME, contents=prompt)
    total_time = time.time() - start_time

    return {
        'answer': response.text,
        'chunks_retrieved': len(retrieved_chunks),
        'context_tokens': len(context.split()),
        'retrieval_time_ms': round(retrieval_time * 1000, 2),
        'total_time_ms': round(total_time * 1000, 2),
    }

print('RAG pipeline ready')

---
# ARCHITECTURE 2: CAG (Cache Augmented Generation)

Preload ENTIRE document → LLM reads everything → No retrieval, no embeddings, no vector DB.

Gemini 2.0 Flash has a 1M token context window. Most internal docs fit easily.

In [ ]:
def cag_query_basic(question):
    start_time = time.time()
    prompt = f'You are a helpful assistant. Answer the question based ONLY on the provided document. If the document does not contain the answer, say I dont have enough information.\n\nFULL DOCUMENT:\n{KNOWLEDGE_BASE}\n\nQUESTION: {question}\n\nANSWER:'
    response = gemini_client.models.generate_content(model=MODEL_NAME, contents=prompt)
    total_time = time.time() - start_time
    return {
        'answer': response.text,
        'context_tokens': len(KNOWLEDGE_BASE.split()),
        'retrieval_time_ms': 0,
        'total_time_ms': round(total_time * 1000, 2),
    }

print('Basic CAG pipeline ready')

In [ ]:
# True CAG with Gemini Context Caching (KV Cache)
from google.genai import types
import datetime

print('Creating cached context (precomputing KV representations)...')
cache_start = time.time()

cache = gemini_client.caches.create(
    model=MODEL_NAME,
    config=types.CreateCachedContentConfig(
        display_name='novamind_knowledge_base',
        system_instruction='You are a helpful assistant. Answer questions based ONLY on the provided document. If the document does not contain the answer, say I dont have enough information.',
        contents=[
            types.Content(
                role='user',
                parts=[types.Part(text=f'Here is the knowledge base document:\n\n{KNOWLEDGE_BASE}')]
            )
        ],
        ttl=datetime.timedelta(minutes=30),
    )
)

cache_creation_time = round((time.time() - cache_start) * 1000, 2)
print(f'Context cached in {cache_creation_time}ms')
print(f'Cache name: {cache.name}')
print(f'Every query now reuses this cached computation.')

In [ ]:
def cag_query_cached(question):
    start_time = time.time()
    response = gemini_client.models.generate_content(
        model=MODEL_NAME,
        contents=question,
        config=types.GenerateContentConfig(cached_content=cache.name)
    )
    total_time = time.time() - start_time
    return {
        'answer': response.text,
        'context_tokens': len(KNOWLEDGE_BASE.split()),
        'retrieval_time_ms': 0,
        'total_time_ms': round(total_time * 1000, 2),
    }

print('Cached CAG pipeline ready')

---
# Baseline: No Context (Hallucination Test)
Since NovaMind is fictional, everything the model says without context should be wrong.

In [ ]:
def baseline_query(question):
    start_time = time.time()
    prompt = f'Answer this question:\n\nQUESTION: {question}\n\nANSWER:'
    response = gemini_client.models.generate_content(model=MODEL_NAME, contents=prompt)
    total_time = time.time() - start_time
    return {
        'answer': response.text,
        'context_tokens': 0,
        'retrieval_time_ms': 0,
        'total_time_ms': round(total_time * 1000, 2),
    }

print('Baseline pipeline ready')

---
# Run the Full Comparison

In [ ]:
results = []

for i, question in enumerate(TEST_QUESTIONS):
    print(f'\n{"="*80}')
    print(f'QUESTION {i+1}: {question}')
    print(f'{"="*80}')

    print('\nBASELINE (No Context):')
    baseline = baseline_query(question)
    print(f'   Time: {baseline["total_time_ms"]}ms')
    ans = baseline['answer']
    print(f'   {ans[:300]}...' if len(ans) > 300 else f'   {ans}')

    print('\nRAG:')
    rag = rag_query(question)
    print(f'   Time: {rag["total_time_ms"]}ms (retrieval: {rag["retrieval_time_ms"]}ms)')
    print(f'   Chunks: {rag["chunks_retrieved"]} ({rag["context_tokens"]} tokens)')
    ans = rag['answer']
    print(f'   {ans[:300]}...' if len(ans) > 300 else f'   {ans}')

    print('\nCAG Basic:')
    cag_b = cag_query_basic(question)
    print(f'   Time: {cag_b["total_time_ms"]}ms')
    ans = cag_b['answer']
    print(f'   {ans[:300]}...' if len(ans) > 300 else f'   {ans}')

    print('\nCAG Cached:')
    cag_c = cag_query_cached(question)
    print(f'   Time: {cag_c["total_time_ms"]}ms')
    ans = cag_c['answer']
    print(f'   {ans[:300]}...' if len(ans) > 300 else f'   {ans}')

    results.append({
        'question': question,
        'baseline': baseline,
        'rag': rag,
        'cag_basic': cag_b,
        'cag_cached': cag_c,
    })

print(f'\nAll {len(TEST_QUESTIONS)} questions processed!')

---
# Performance Charts

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0f1117')

for ax in axes:
    ax.set_facecolor('#1a1d27')
    ax.tick_params(colors='#8b8fa3')
    ax.xaxis.label.set_color('#e8e9ed')
    ax.yaxis.label.set_color('#e8e9ed')
    ax.title.set_color('#e8e9ed')
    for spine in ax.spines.values():
        spine.set_color('#2d3148')

qlabels = [f'Q{i+1}' for i in range(len(TEST_QUESTIONS))]
x = np.arange(len(qlabels))
w = 0.2

bt = [r['baseline']['total_time_ms'] for r in results]
rt = [r['rag']['total_time_ms'] for r in results]
cbt = [r['cag_basic']['total_time_ms'] for r in results]
cct = [r['cag_cached']['total_time_ms'] for r in results]

axes[0].bar(x - 1.5*w, bt, w, label='Baseline', color='#ff6b6b', alpha=0.85)
axes[0].bar(x - 0.5*w, rt, w, label='RAG', color='#00d4aa', alpha=0.85)
axes[0].bar(x + 0.5*w, cbt, w, label='CAG (Basic)', color='#5b9cf6', alpha=0.85)
axes[0].bar(x + 1.5*w, cct, w, label='CAG (Cached)', color='#f5c542', alpha=0.85)
axes[0].set_xlabel('Question')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Response Latency per Question', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(qlabels)
axes[0].legend(facecolor='#1a1d27', edgecolor='#2d3148', labelcolor='#e8e9ed')
axes[0].grid(axis='y', alpha=0.15, color='#8b8fa3')

methods = ['Baseline', 'RAG', 'CAG\n(Basic)', 'CAG\n(Cached)']
avgs = [np.mean(bt), np.mean(rt), np.mean(cbt), np.mean(cct)]
colors = ['#ff6b6b', '#00d4aa', '#5b9cf6', '#f5c542']
bars = axes[1].bar(methods, avgs, color=colors, alpha=0.85)
for bar, val in zip(bars, avgs):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
                f'{val:.0f}ms', ha='center', va='bottom', fontweight='bold',
                fontsize=11, color='#e8e9ed')
axes[1].set_ylabel('Average Latency (ms)')
axes[1].set_title('Average Latency Comparison', fontweight='bold')
axes[1].grid(axis='y', alpha=0.15, color='#8b8fa3')

plt.tight_layout()
plt.savefig('rag_vs_cag_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Chart saved as rag_vs_cag_comparison.png')

In [ ]:
# Summary table
import numpy as np
print('\n' + '='*80)
print('ARCHITECTURE COMPARISON SUMMARY')
print('='*80)
print(f'''
                        RAG          CAG (Basic)    CAG (Cached)
Avg Latency          {np.mean(rt):>8.0f} ms   {np.mean(cbt):>8.0f} ms     {np.mean(cct):>8.0f} ms
Retrieval Step          Yes            No              No
Vector DB            Required       Not needed      Not needed
Embedding Model      Required       Not needed      Not needed
Scales to 100s docs    Yes            No              No
Accuracy Risk        Wrong chunks     Minimal         Minimal
Best For             Large KB       Small KB       Repeated queries
''')

In [ ]:
# Cleanup cache
try:
    gemini_client.caches.delete(name=cache.name)
    print('Cache deleted')
except Exception as e:
    print(f'Cache cleanup: {e}')

---
# When Do You Use Which?

**RAG:** Large knowledge bases, constantly updated docs, need source attribution

**CAG:** Fits in context window, need complete context, simplicity matters, repeated queries

**Both:** RAG retrieves candidate docs, CAG processes them with full context. This is where production systems are heading.

---

**The model did not get smarter. The architecture did.**

*Built by Anubhav Singhmaar | Cost: Rs 0 | Stack: Gemini 2.0 Flash, ChromaDB, Sentence Transformers, Google Colab*